# Arbitrage Calculator: Cloud GPU ML Matcher (v5)
### Qwen3-Embedding-0.6B + bge-reranker-v2-m3 + Qwen3-Reranker-4B | pure torch on T4 | WebSocket pipeline

### Instructions
1. **Runtime > Change runtime type** -> ensure **GPU T4 x2** (dual T4) is selected
2. **Runtime > Run all** (or Ctrl+F9). Auto-execute will trigger as soon as the runtime connects.

### What changed from v4
- Stage 1 bi-encoder: bge-m3 fp32 -> **Qwen/Qwen3-Embedding-0.6B fp16** (better short-text similarity, 4x less VRAM)
- Stage 3: Gemma-4 27B 4-bit (model id did not exist; never ran) -> **Qwen/Qwen3-Reranker-4B fp16** (yes-token logit, fits one T4)
- **Polarity handling**: stage 3 scores Same vs Inverted; inverted pairs become YES+YES arbs instead of false matches
- **Arb math fixed**: costs now use the tradeable side of the book (ask to buy YES, 1-bid to buy NO)
- Casefolded proper-noun veto (ForecastEx ALL-CAPS titles), number unit normalization ($1.5M == 1,500,000), endDate proximity veto
- WS robustness: typed-message recv loop, status-poll fail-fast, local results backup + HTTP POST fallback

### What is intentionally NOT here
- No `faiss-gpu` (pure torch `util.cos_sim` - bulletproof on T4)
- No flash-attention (T4 is sm_75)
- No bitsandbytes (fp16 everywhere, no 4-bit needed)

In [ ]:
# __BEACON__ live progress reporting (injected by executor; no-op if URL unset)
import urllib.request as _ulr, json as _json, threading as _thr
_BEACON_URL = "BEACON_URL_PLACEHOLDER"
def _beacon(stage, status, message=""):
    if not _BEACON_URL or "PLACEHOLDER" in _BEACON_URL:
        return
    def _s():
        try:
            _d = _json.dumps({"stage": stage, "status": status, "message": message}).encode()
            _rq = _ulr.Request(_BEACON_URL, data=_d,
                               headers={"Content-Type": "application/json"}, method="POST")
            _ulr.urlopen(_rq, timeout=5).read()
        except Exception:
            pass
    try:
        _thr.Thread(target=_s, daemon=True).start()
    except Exception:
        pass

globals().get('_beacon', lambda *a, **k: None)(0, 'running', 'Installing deps + GPU check')
# 1. Install + GPU check
# Kaggle's preinstalled torch matches the image CUDA — do NOT reinstall it.
!pip install -q sentence-transformers httpx websockets pydantic nest_asyncio \
             rapidfuzz spacy 'transformers>=4.51.0'
!python -m spacy download en_core_web_sm -q

import torch, os, math
from sentence_transformers import SentenceTransformer, CrossEncoder, util
from transformers import AutoTokenizer, AutoModelForCausalLM
from concurrent.futures import ThreadPoolExecutor
import httpx, websockets, asyncio, json, re, time
from typing import List, Dict, Any, Set, Tuple
import nest_asyncio
from rapidfuzz import fuzz as _rfuzz
import spacy as _spacy
nest_asyncio.apply()


# ── Dual T4 GPU assertion ───────────────────────────────────────────────
_num_gpus = torch.cuda.device_count()
if _num_gpus < 2:
    raise RuntimeError(
        f'Expected 2 GPUs (dual T4) but found {_num_gpus}. '
        'Go to Runtime > Change runtime type and select "GPU T4 x2".'
    )
print(f'✓ Dual T4 GPUs confirmed ({_num_gpus} devices)')

# HuggingFace auth (injected at push time or from Kaggle Secrets)
try:
    from kaggle_secrets import UserSecretsClient
    _HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF token from Kaggle Secrets')
except Exception:
    _HF_TOKEN = os.environ.get('HF_TOKEN', 'HF_TOKEN_PLACEHOLDER')
    print(f'HF token from env ({"set" if _HF_TOKEN and _HF_TOKEN != "HF_TOKEN_PLACEHOLDER" else "missing"})')

if _HF_TOKEN and _HF_TOKEN != "HF_TOKEN_PLACEHOLDER":
    try:
        from huggingface_hub import login as _hf_login
        _hf_login(token=_HF_TOKEN, add_to_git_credential=False)
        print("HuggingFace Hub authenticated.")
    except Exception as _le:
        print(f"HF login failed: {_le}")

print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for _i in range(torch.cuda.device_count()):
        _p = torch.cuda.get_device_properties(_i)
        print(f'  GPU {_i}: {_p.name} ({_p.total_memory/1e9:.1f} GB)')

try:
    _ng = torch.cuda.device_count()
    if _ng:
        _nm = torch.cuda.get_device_properties(0).name
        _gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        _gpu_msg = f'{_ng}x {_nm} ({_gb:.0f}GB each)'
    else:
        _gpu_msg = 'WARNING: no GPU allocated'
except Exception:
    _gpu_msg = 'GPU check unavailable'
globals().get('_beacon', lambda *a, **k: None)(0, 'done', _gpu_msg)


In [ ]:
globals().get('_beacon', lambda *a, **k: None)(1, 'running', 'Loading Qwen3-Embedding + rerankers onto T4(s)')
# 2. Load models onto T4(s) — Qwen3-Embedding-0.6B + bge-reranker-v2-m3 + Qwen3-Reranker-4B
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0

print(f'Device: {DEVICE} | GPUs: {NUM_GPUS}')
for _i in range(NUM_GPUS):
    _p = torch.cuda.get_device_properties(_i)
    print(f'  GPU {_i}: {_p.name} ({_p.total_memory/1e9:.1f} GB)')

# ── Stage-1 bi-encoder: Qwen3-Embedding-0.6B fp16 (~1.2 GB on cuda:0) ────────
print('Loading bi-encoder   (Qwen/Qwen3-Embedding-0.6B, fp16)...')
bi_model = SentenceTransformer(
    'Qwen/Qwen3-Embedding-0.6B', device=DEVICE,
    model_kwargs={'dtype': torch.float16},
    processor_kwargs={'padding_side': 'left'},
)
_encode_devices = [f'cuda:{i}' for i in range(NUM_GPUS)] if NUM_GPUS > 1 else None
_mp_pool = bi_model.start_multi_process_pool(_encode_devices) if _encode_devices else None
if _mp_pool: print(f'  bi-encoder pool: {_encode_devices}')

def encode_titles(titles, batch_size=256):
    if _mp_pool:
        # bound call — encode(self, sentences, pool, ...)
        embs = bi_model.encode(
            titles, pool=_mp_pool, batch_size=batch_size, normalize_embeddings=True)
        return torch.tensor(embs, device=DEVICE)
    return bi_model.encode(titles, convert_to_tensor=True, batch_size=batch_size,
                           normalize_embeddings=True, show_progress_bar=True)

# ── Stage-2 reranker: bge-reranker-v2-m3 (fast cross-encoder, cuda:0) ────────
print('Loading reranker     (BAAI/bge-reranker-v2-m3)...')
reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', device=DEVICE)

# ── Stage-3 deep reranker: Qwen3-Reranker-4B fp16 (~8 GB) ───────────────────
# Yes-token-logit scoring per the official model card. Lives on the LAST GPU so
# cuda:0 keeps stage-1/2 plus the multiprocess pool headroom.
_QR_ID  = 'Qwen/Qwen3-Reranker-4B'
_QR_DEV = f'cuda:{NUM_GPUS-1}' if NUM_GPUS else 'cpu'
_qr_tok, _qr_mdl = None, None
try:
    print(f'Loading deep-reranker ({_QR_ID}, fp16, {_QR_DEV})...')
    _qr_tok = AutoTokenizer.from_pretrained(_QR_ID, padding_side='left')
    _qr_mdl = AutoModelForCausalLM.from_pretrained(
        _QR_ID, dtype=torch.float16).to(_QR_DEV).eval()
    print(f'  deep-reranker ready ({torch.cuda.memory_allocated(NUM_GPUS-1)/1e9:.1f} GB on {_QR_DEV})' if NUM_GPUS else '  deep-reranker on CPU')
except Exception as _e:
    print(f'WARN: deep-reranker load failed ({_e}). Stage-3 disabled; stage-2 order kept.')
    _qr_tok, _qr_mdl = None, None

_QR_PREFIX = ('<|im_start|>system\nJudge whether the Document meets the requirements '
              'based on the Query and the Instruct provided. Note that the answer can '
              'only be "yes" or "no".<|im_end|>\n<|im_start|>user\n')
_QR_SUFFIX = '<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'
if _qr_tok:
    _QR_NO  = _qr_tok.convert_tokens_to_ids('no')
    _QR_YES = _qr_tok.convert_tokens_to_ids('yes')

def qwen_rerank_scores(pairs, instruction, batch_size=8, max_length=512):
    """P('yes') for each (title_a, title_b) pair under the given instruction."""
    if not _qr_mdl:
        return [0.0] * len(pairs)
    out = []
    for i in range(0, len(pairs), batch_size):
        batch = pairs[i:i+batch_size]
        texts = [f'{_QR_PREFIX}<Instruct>: {instruction}\n<Query>: {a[:300]}\n<Document>: {b[:300]}{_QR_SUFFIX}'
                 for a, b in batch]
        inp = _qr_tok(texts, return_tensors='pt', padding=True,
                      truncation=True, max_length=max_length).to(_QR_DEV)
        with torch.no_grad():
            logits = _qr_mdl(**inp).logits[:, -1, :]
        sel = logits[:, [_QR_NO, _QR_YES]].float().log_softmax(dim=-1)
        out.extend(sel[:, 1].exp().tolist())
    return out

# ── spaCy NER — entities precomputed ONCE per title (was: per pair) ──────────
try:
    _nlp = _spacy.load('en_core_web_sm', disable=['parser','tagger','lemmatizer'])
    _ENT_LABELS = ('PERSON','ORG','GPE','EVENT','NORP','FAC')
    def build_entity_cache(titles):
        cache = {}
        for title, doc in zip(titles, _nlp.pipe(titles, batch_size=512)):
            cache[title] = {e.text.casefold() for e in doc.ents if e.label_ in _ENT_LABELS}
        return cache
    print('spaCy NER ready (batch pipe).')
except Exception as _e:
    print(f'WARN: spaCy load failed ({_e}). NER disabled.')
    def build_entity_cache(titles): return {t: set() for t in titles}

print('All models loaded.')

globals().get('_beacon', lambda *a, **k: None)(1, 'done', '')


In [ ]:
globals().get('_beacon', lambda *a, **k: None)(2, 'running', 'Fetching markets over WebSocket + pre-filter')
# 3. WebSocket fetch + binary/noise pre-filter
# The backend rewrites the placeholder line into `WS_URL = "wss://...your-ngrok..."` before upload.
WS_URL_PLACEHOLDER = "REPLACE_ME"
WS_URL = WS_URL_PLACEHOLDER

# ── Typed-message receive: the backend broadcasts state frames to ALL clients,
#    so the next recv() is NOT guaranteed to be the reply we want. Loop until
#    the expected type arrives (ignoring broadcasts) or the deadline passes.
async def recv_expected(ws, expected_type, deadline=120):
    end = time.time() + deadline
    while time.time() < end:
        remaining = max(1.0, end - time.time())
        try:
            raw = await asyncio.wait_for(ws.recv(), timeout=remaining)
        except asyncio.TimeoutError:
            break
        try:
            msg = json.loads(raw)
        except Exception:
            continue
        if msg.get('type') == expected_type:
            return msg
        # else: broadcast frame — ignore and keep waiting
    raise TimeoutError(f'No {expected_type} message within {deadline}s')

# ── Polite scan-status polling ───────────────────────────────────────────────
# VITAL RULE: this notebook NEVER interrupts scanning and NEVER pulls partial
# data. It politely asks every 60s until the backend says every scan (incl.
# both IBKR rounds) is fully finished. If the scans never finish, or the
# backend errors, we ABORT WITHOUT PULLING — a report built on incomplete
# market data is worse than no report.
# Fail-fast applies only to a tunnel routed to the WRONG service (responses
# that are not the arbitrage API at all).
import time as _time
_backend_http = WS_URL.replace('wss://', 'https://').replace('/ws', '')
print(f'[status-poll] backend = {_backend_http}')

_MAX_WAIT       = 4 * 3600   # generous ceiling; on expiry we ABORT, never pull partial
_POLL_SEC       = 60
_MAX_CONSEC_ERR = 10
_waited         = 0
_consec_err     = 0

while _waited < _MAX_WAIT:
    try:
        with httpx.Client(timeout=15) as _c:
            _r = _c.get(f'{_backend_http}/api/scan-status')
        _st = _r.json()
        _consec_err = 0
        _status      = _st.get('status', 'unknown')
        _phase       = _st.get('phase', '')
        _ibkr_rounds = _st.get('ibkr_scan_rounds_done', 0)
        _total       = _st.get('total_markets', 0)
        _msg         = _st.get('message', '')[:80]
        print(f'  [{_waited:>5}s] {_status} | {_phase} | markets={_total:,} | IBKR_rounds={_ibkr_rounds} | {_msg}')

        # Proceed ONLY when IBKR has done both rounds AND scan is fully finished
        _scan_done = _status in ('complete', 'idle', 'waiting_for_cloud')
        if _ibkr_rounds >= 2 and _scan_done and _total > 0:
            print(f'  ✓ All scans complete ({_ibkr_rounds} IBKR rounds, {_total:,} markets). Fetching...')
            break
        # Legacy backends that don't track ibkr_rounds
        if _scan_done and _total > 10000 and _ibkr_rounds == 0:
            print(f'  ✓ Scan complete ({_total:,} markets, legacy). Fetching...')
            break
        if _status == 'error':
            raise RuntimeError(
                f'Backend scan errored ({_msg}) — aborting WITHOUT pulling partial data. '
                'Fix the scan and re-run this notebook.')
    except RuntimeError:
        raise
    except Exception as _e:
        _consec_err += 1
        print(f'  Status check error ({_consec_err}/{_MAX_CONSEC_ERR}): {_e}')
        if _consec_err >= _MAX_CONSEC_ERR:
            raise RuntimeError(
                'Backend URL is not serving the arbitrage API — check ngrok routing '
                f'({_backend_http}/api/scan-status unparseable {_MAX_CONSEC_ERR}x in a row)')

    _time.sleep(_POLL_SEC)
    _waited += _POLL_SEC
else:
    raise RuntimeError(
        f'Scans still not finished after {_MAX_WAIT/3600:.0f}h — aborting WITHOUT pulling '
        'partial data. Check the backend scan, then re-run this notebook.')

NOISE_PATTERNS = re.compile(
    r'exact score|\bmap \d+\b|\bround \d+\b|\bset \d+\b|\bhalf \d+\b'
    r'|over/under|o/u \d|\+/-|handicap|total kills|total goals'
    r'|first blood|first dragon|odd/even|both teams to score',
    re.IGNORECASE
)

def is_binary(m):
    if not m.get('isBinary', True): return False
    if m.get('outcomeCount', 2) > 2: return False
    return True

async def fetch_markets_via_websocket():
    print(f'Connecting to backend: {WS_URL}')
    async with websockets.connect(WS_URL, max_size=64*1024*1024) as ws:
        await ws.send(json.dumps({'type': 'subscribe', 'channel': 'markets'}))
        data = await recv_expected(ws, 'markets_data', deadline=180)
        return data.get('markets', [])

all_markets_raw = asyncio.get_event_loop().run_until_complete(fetch_markets_via_websocket())
print(f'Raw markets received: {len(all_markets_raw):,}')

_malformed = 0
seen, unique = set(), []
for m in all_markets_raw:
    if not isinstance(m, dict) or not m.get('title') or not m.get('platform'):
        _malformed += 1
        continue
    key = m.get('id') or m.get('marketUrl')
    if key not in seen:
        seen.add(key); unique.append(m)
print(f'After dedup        : {len(unique):,} ({_malformed} malformed dropped)')

binary = [m for m in unique if is_binary(m)]
print(f'Binary only        : {len(binary):,} (dropped {len(unique)-len(binary):,} multi-outcome)')

clean = [m for m in binary if not NOISE_PATTERNS.search(m.get('title',''))]
print(f'After noise filter : {len(clean):,} (dropped {len(binary)-len(clean):,} sports/gaming noise)')

by_platform_preview = {}
for m in clean:
    by_platform_preview.setdefault(m.get('platform','?'), []).append(m)
for plat, mkts in by_platform_preview.items():
    print(f'  {plat:12} {len(mkts):,}')

all_markets = clean

globals().get('_beacon', lambda *a, **k: None)(2, 'done', '')


In [ ]:
globals().get('_beacon', lambda *a, **k: None)(3, 'running', 'Building compatibility filters')
# 4. Compatibility filters: dates / numbers (unit-normalized) / proper nouns (casefolded) / endDate
from datetime import datetime

PROPER_NOUN_RE = re.compile(r"\b([A-Z][a-zA-Z]{2,})\b")
DATE_RE_LIST = [
    re.compile(r'\b(\d{1,2})/(\d{1,2})/(\d{4})\b'),
    re.compile(r'\b(\d{4})-(\d{1,2})-(\d{1,2})\b'),
    re.compile(r'\b(January|February|March|April|May|June|July|August|September|October|November|December)\s+(\d{1,2})(?:st|nd|rd|th)?(?:,?\s+(\d{4}))?\b', re.IGNORECASE),
]
NUMBER_RE = re.compile(r'\d+(?:\.\d+)?')
STOPWORDS = {'will','the','be','by','at','in','on','for','of','to','and','or','a','an','is','are','was','were','yes','no'}

# number-unit normalization: "$1.5 million" == "1,500,000" == "1.5M"
_THOUSANDS_SEP_RE = re.compile(r'(?<=\d),(?=\d{3}\b)')
_SUFFIX_NUM_RE = re.compile(r'(\d+(?:\.\d+)?)\s*(k|thousand|m|mn|million|b|bn|billion)\b', re.IGNORECASE)
_SUFFIX_MULT = {'k':1e3,'thousand':1e3,'m':1e6,'mn':1e6,'million':1e6,'b':1e9,'bn':1e9,'billion':1e9}

def extract_dates(text: str) -> Set[Tuple[int,int,int]]:
    out = set()
    for pat in DATE_RE_LIST:
        for m in pat.finditer(text):
            try:
                if pat is DATE_RE_LIST[0]:
                    mo, d, y = int(m.group(1)), int(m.group(2)), int(m.group(3))
                    out.add((y, mo, d))
                elif pat is DATE_RE_LIST[1]:
                    y, mo, d = int(m.group(1)), int(m.group(2)), int(m.group(3))
                    out.add((y, mo, d))
                else:
                    months = {'january':1,'february':2,'march':3,'april':4,'may':5,'june':6,'july':7,'august':8,'september':9,'october':10,'november':11,'december':12}
                    mo = months[m.group(1).lower()]; d = int(m.group(2))
                    y = int(m.group(3)) if m.group(3) else 0
                    if y: out.add((y, mo, d))
            except Exception:
                pass
    return out

def extract_numbers(text: str) -> Set[float]:
    text = _THOUSANDS_SEP_RE.sub('', text)   # 1,500,000 -> 1500000
    out, consumed = set(), []
    for m in _SUFFIX_NUM_RE.finditer(text):  # 1.5M -> 1500000.0
        out.add(float(m.group(1)) * _SUFFIX_MULT[m.group(2).lower()])
        consumed.append(m.span(1))
    for m in NUMBER_RE.finditer(text):
        if any(s <= m.start() < e for s, e in consumed):
            continue
        v = float(m.group())
        if not (2020 <= v <= 2030):          # skip bare years
            out.add(v)
    return out

def _numbers_match(a: float, b: float) -> bool:
    # absolute tolerance for small numbers, 1% relative for large ones
    return abs(a - b) <= max(0.5, 0.01 * max(abs(a), abs(b)))

def extract_proper_nouns(text: str) -> Set[str]:
    # casefolded so ForecastEx ALL-CAPS ("FED") matches mixed case ("Fed")
    return {w.casefold() for w in PROPER_NOUN_RE.findall(text)
            if w.casefold() not in STOPWORDS}

def _parse_end(s):
    if not s: return None
    try:
        return datetime.fromisoformat(str(s).replace('Z', '+00:00')).timestamp()
    except Exception:
        return None

def end_dates_compatible(ma: dict, mb: dict, max_days: float = 45) -> bool:
    """Veto pairs whose market endDates are more than max_days apart.
    Far stronger than title regex — every market carries endDate."""
    ta, tb = _parse_end(ma.get('endDate')), _parse_end(mb.get('endDate'))
    if ta is None or tb is None:
        return True
    return abs(ta - tb) <= max_days * 86400

def are_compatible(title_a: str, title_b: str) -> Tuple[bool, str]:
    da, db = extract_dates(title_a), extract_dates(title_b)
    if da and db and da.isdisjoint(db):
        return False, 'date mismatch'
    na, nb = extract_numbers(title_a), extract_numbers(title_b)
    if na and nb and not any(_numbers_match(a, b) for a in na for b in nb):
        return False, 'number mismatch'
    pa, pb = extract_proper_nouns(title_a), extract_proper_nouns(title_b)
    if pa and pb and pa.isdisjoint(pb):
        return False, 'proper-noun mismatch'
    return True, 'compatible'

print('Compatibility filters ready (date / unit-normalized number / casefolded proper-noun / endDate).')

globals().get('_beacon', lambda *a, **k: None)(3, 'done', '')


In [ ]:
globals().get('_beacon', lambda *a, **k: None)(4, 'running', 'GPU matching: embed -> rerank -> rerank')
# 5. GPU matching: Qwen3-Embedding -> bge-reranker-v2-m3 -> Qwen3-Reranker-4B (3-stage, polarity-aware)

_SAME_INSTR = ('Determine if these two prediction market questions resolve YES under '
               'the same real-world outcome (they are the same bet).')
_INV_INSTR  = ('Determine if these two prediction market questions resolve in OPPOSITE '
               'directions for the same real-world event (one resolves YES exactly when '
               'the other resolves NO).')

def compute_pair_arb(ma, mb, inverted=False):
    """Cost uses the TRADEABLE side of the book: buying YES costs the ask,
    buying NO costs (1 - bid). The old version used bid+(1-ask) — the cheap,
    untradeable side — overstating ROI by the full spread on each leg."""
    yes_a = ma.get('yesPrice', 0.5); yes_b = mb.get('yesPrice', 0.5)
    ask_a = ma.get('bestAsk', yes_a); bid_a = ma.get('bestBid', yes_a)
    ask_b = mb.get('bestAsk', yes_b); bid_b = mb.get('bestBid', yes_b)

    if inverted:
        # inverted pair: YES on A and YES on B cover complementary outcomes
        cost = ask_a + ask_b
        roi  = ((1 - cost) / max(0.01, cost) * 100) if 0.05 < cost < 1 else -100
        return {'roi': min(1000, roi), 'cost': cost, 'scenario': 3}

    cost1 = ask_a + (1 - bid_b)        # buy YES on A + buy NO on B
    roi1  = ((1 - cost1) / max(0.01, cost1) * 100) if 0.05 < cost1 < 1 else -100
    cost2 = (1 - bid_a) + ask_b        # buy NO on A + buy YES on B
    roi2  = ((1 - cost2) / max(0.01, cost2) * 100) if 0.05 < cost2 < 1 else -100
    if roi1 >= roi2:
        return {'roi': min(1000, roi1), 'cost': cost1, 'scenario': 1}
    return {'roi': min(1000, roi2), 'cost': cost2, 'scenario': 2}

def _fuzzy_score(a: str, b: str) -> float:
    """rapidfuzz token sort ratio — 0-100."""
    return _rfuzz.token_sort_ratio(a, b)

def match_markets(markets, top_k=2000, min_similarity=0.62, min_roi=0.1,
                  candidate_top_k=50):
    t0 = time.time()
    by_platform = {}
    _skipped = 0
    for m in markets:
        plat = m.get('platform')
        if not plat or m.get('yesPrice') is None:
            _skipped += 1
            continue
        by_platform.setdefault(plat, []).append(m)
    platforms = list(by_platform.keys())
    print(f'Platforms: {platforms} ({_skipped} malformed skipped)')

    # entity sets precomputed ONCE per title (was: per candidate pair)
    _all_titles = [m['title'] for mkts in by_platform.values() for m in mkts]
    print(f'Precomputing NER entities for {len(_all_titles):,} titles...')
    ent_cache = build_entity_cache(_all_titles)

    def _entity_compatible(ta: str, tb: str) -> bool:
        ea, eb = ent_cache.get(ta, set()), ent_cache.get(tb, set())
        return not (ea and eb and ea.isdisjoint(eb))

    # ── Stage 1: bi-encoder cosine similarity matrix ─────────────────────────
    plat_embeddings = {}
    for plat in platforms:
        titles = [m['title'] for m in by_platform[plat]]
        print(f'  Encoding {len(titles):,} markets for {plat}...')
        plat_embeddings[plat] = encode_titles(titles)

    candidates = []
    for i in range(len(platforms)):
        pa = platforms[i]; emb_a = plat_embeddings[pa]; mkts_a = by_platform[pa]
        for j in range(i+1, len(platforms)):
            pb = platforms[j]; emb_b = plat_embeddings[pb]; mkts_b = by_platform[pb]
            print(f'  cos_sim {pa} x {pb} ({emb_a.shape[0]:,} x {emb_b.shape[0]:,})')
            scores = util.cos_sim(emb_a, emb_b)
            top    = torch.topk(scores, k=min(candidate_top_k, scores.shape[1]), dim=1)
            pair_count = 0
            for ai in range(top.values.shape[0]):
                ma = mkts_a[ai]
                for s_val, bi in zip(top.values[ai].cpu().tolist(), top.indices[ai].cpu().tolist()):
                    if s_val < min_similarity: break
                    mb = mkts_b[int(bi)]
                    ok, reason = are_compatible(ma['title'], mb['title'])
                    if not ok: continue
                    if not end_dates_compatible(ma, mb): continue
                    if not _entity_compatible(ma['title'], mb['title']): continue
                    # keep the pair if EITHER polarity is profitable — stage 3 decides which
                    arb_same = compute_pair_arb(ma, mb)
                    arb_inv  = compute_pair_arb(ma, mb, inverted=True)
                    if max(arb_same['roi'], arb_inv['roi']) < min_roi: continue
                    fuzzy = _fuzzy_score(ma['title'], mb['title'])
                    candidates.append((ma, mb, arb_same, arb_inv, float(s_val), fuzzy))
                    pair_count += 1
            print(f'    {pair_count:,} candidates')

    if not candidates:
        print('No candidates.'); return []

    # Dedup + sort by composite (bi_score + fuzzy_bonus)
    seen, deduped = set(), []
    for c in sorted(candidates, key=lambda x: x[4] + x[5]/200.0, reverse=True):
        key = tuple(sorted([c[0].get('id',''), c[1].get('id','')]))
        if key in seen: continue
        seen.add(key); deduped.append(c)

    # ── Stage 2: bge-reranker-v2-m3 (fast cross-encoder) ─────────────────────
    stage2_pool = deduped[:top_k * 4]
    print(f'Stage-2 reranking {len(stage2_pool):,} candidates (bge-reranker-v2-m3)...')
    pairs_s2 = [[c[0]['title'], c[1]['title']] for c in stage2_pool]
    s2_scores = reranker.predict(pairs_s2, show_progress_bar=True, batch_size=64)

    stage2_out = []
    for score, cand in zip(s2_scores, stage2_pool):
        if float(score) <= 0.3: continue
        ma, mb, arb_same, arb_inv, bi_score, fuzzy = cand
        stage2_out.append({
            'ma': ma, 'mb': mb, 'arb_same': arb_same, 'arb_inv': arb_inv,
            'bi_score': bi_score, 's2_score': float(score), 'fuzzy': fuzzy,
            'inverted': False,
        })
    stage2_out.sort(key=lambda x: x['s2_score'] + x['fuzzy']/200.0, reverse=True)
    print(f'Stage-2 survivors: {len(stage2_out):,}')

    # ── Stage 3: Qwen3-Reranker-4B, polarity-aware (Same vs Inverted) ────────
    stage3_input = stage2_out[:500]
    if _qr_mdl and stage3_input:
        print(f'Stage-3 deep-reranking {len(stage3_input)} via Qwen3-Reranker-4B (same + inverted passes)...')
        pairs_s3 = [(c['ma']['title'], c['mb']['title']) for c in stage3_input]
        same_scores = qwen_rerank_scores(pairs_s3, _SAME_INSTR)
        inv_scores  = qwen_rerank_scores(pairs_s3, _INV_INSTR)
        n_inverted = 0
        for c, s_same, s_inv in zip(stage3_input, same_scores, inv_scores):
            if s_inv > s_same and s_inv > 0.6:
                c['inverted'] = True
                c['s3_score'] = float(s_inv)
                n_inverted += 1
            else:
                c['s3_score'] = float(s_same)
        print(f'  {n_inverted} inverted pairs detected (YES+YES arbs)')
        stage3_input.sort(key=lambda x: x.get('s3_score', 0), reverse=True)
        s3_ids = {tuple(sorted([c['ma'].get('id',''), c['mb'].get('id','')])) for c in stage3_input}
        stage2_rem = [c for c in stage2_out[500:] if tuple(sorted([c['ma'].get('id',''), c['mb'].get('id','')])) not in s3_ids]
        merged = stage3_input + stage2_rem
    else:
        merged = stage2_out

    final = []
    for c in merged[:top_k]:
        ma, mb = c['ma'], c['mb']
        arb = c['arb_inv'] if c.get('inverted') else c['arb_same']
        s3 = c.get('s3_score', c['s2_score'])
        ya = ma.get('yesPrice', 0.5); yb = mb.get('yesPrice', 0.5)
        final.append({
            'marketA':       {'id': ma.get('id'), 'platform': ma.get('platform','?'), 'title': ma['title'],
                              'marketUrl': ma.get('marketUrl',''), 'yesPrice': ya,
                              'noPrice': ma.get('noPrice', round(1-ya,4)),
                              'endDate': ma.get('endDate')},
            'marketB':       {'id': mb.get('id'), 'platform': mb.get('platform','?'), 'title': mb['title'],
                              'marketUrl': mb.get('marketUrl',''), 'yesPrice': yb,
                              'noPrice': mb.get('noPrice', round(1-yb,4)),
                              'endDate': mb.get('endDate')},
            'roi':           arb['roi'],
            'scenario':      arb['scenario'],
            'inverted':      bool(c.get('inverted', False)),
            'matchScore':    round(s3 * 100, 1),
            'biEncoderScore':round(c['bi_score'], 4),
            'fuzzyScore':    round(c['fuzzy'], 1),
            'stage2Score':   round(c['s2_score'], 4),
            'stage3Score':   round(c.get('s3_score', c['s2_score']), 4),
            'isVerified':    s3 > 0.80,
        })

    final.sort(key=lambda x: (x['isVerified'], x['matchScore'], x['roi']), reverse=True)
    print(f'Done in {time.time()-t0:.1f}s. {len(final):,} pass all 3 stages.')
    return final

found_pairs = match_markets(all_markets, top_k=2000, min_similarity=0.62, min_roi=0.1)
print(f'\nFinal: {len(found_pairs):,} arbitrage pairs.')

globals().get('_beacon', lambda *a, **k: None)(4, 'done', '')


In [ ]:
globals().get('_beacon', lambda *a, **k: None)(5, 'running', 'Sending results back to backend')
# 6. Send results back: local backup -> WS with retries -> HTTP POST fallback
_BACKUP_PATH = '/kaggle/working/matches_backup.json'

async def send_results_via_websocket(pairs):
    print(f'Sending {len(pairs):,} pairs back to backend over WS...')
    async with websockets.connect(WS_URL, max_size=64*1024*1024) as ws:
        await ws.send(json.dumps({
            'type': 'cloud_results',
            'data': pairs,
            'count': len(pairs),
            'timestamp': time.time(),
        }))
        ack = await recv_expected(ws, 'results_received', deadline=120)
        print(f'Backend ack: {ack.get("message","")}')
        return True

def send_results_via_http(pairs):
    http_base = WS_URL.replace('wss://', 'https://').replace('/ws', '')
    print(f'HTTP fallback: POST {http_base}/api/cloud-results ...')
    with httpx.Client(timeout=120) as c:
        r = c.post(f'{http_base}/api/cloud-results', json=pairs)
        r.raise_for_status()
        print(f'HTTP fallback OK: {r.json()}')
        return True

if found_pairs:
    # 1. Local backup FIRST — GPU work survives any network failure
    try:
        with open(_BACKUP_PATH, 'w') as f:
            json.dump(found_pairs, f)
        print(f'Backup saved: {_BACKUP_PATH} ({len(found_pairs):,} pairs)')
    except Exception as e:
        print(f'WARN: backup write failed: {e}')

    # 2. WS push with retries, then HTTP fallback
    ok = False
    for attempt in range(1, 4):
        try:
            ok = asyncio.get_event_loop().run_until_complete(
                send_results_via_websocket(found_pairs))
            break
        except Exception as e:
            print(f'WS push attempt {attempt}/3 failed: {e}')
            time.sleep(5 * attempt)
    if not ok:
        try:
            ok = send_results_via_http(found_pairs)
        except Exception as e:
            print(f'HTTP fallback also failed: {e}')
            print(f'Results are preserved in {_BACKUP_PATH} — download from the Kaggle output tab.')
    if ok:
        print('Pipeline complete - results returned through the ngrok tunnel.')
else:
    print('No pairs to send.')

globals().get('_beacon', lambda *a, **k: None)(5, 'done', '')


In [ ]:
globals().get('_beacon', lambda *a, **k: None)(6, 'running', 'Previewing top opportunities')
# 7. Preview top opportunities
print('=' * 90)
print(f'TOP {min(10, len(found_pairs))} ARBITRAGE OPPORTUNITIES')
print('=' * 90)
for i, p in enumerate(found_pairs[:10], 1):
    badge = 'VERIFIED' if p['isVerified'] else 'review'
    inv   = '  [INVERTED: buy YES on both]' if p.get('inverted') else ''
    print(f"\n#{i}  ROI: {p['roi']:.2f}%  match: {p['matchScore']}%  ({badge}){inv}")
    print(f"  [A] {p['marketA']['platform']:12}  {p['marketA']['title']}")
    print(f"        YES {p['marketA']['yesPrice']:.3f}  NO {p['marketA']['noPrice']:.3f}")
    print(f"  [B] {p['marketB']['platform']:12}  {p['marketB']['title']}")
    print(f"        YES {p['marketB']['yesPrice']:.3f}  NO {p['marketB']['noPrice']:.3f}")

globals().get('_beacon', lambda *a, **k: None)(6, 'done', '')
